# Saddle-Node bifurcation model for Pain

For an organism which has the ability to move around, a perceived opportunity (threat) can lead to a gain (loss) if it is realized. The core theory of Edmund Rolls' theory of emotions is that genes do not hardcode behavioral actions; they encode "rewards and punishers" to exploit opportunities and to avoid threats. These "rewards and punishers" are implemented in the orbitofrontal cortext (OFC). The function of the orbitofrontal cortex in emotion is to represent these rewards and punishers: "Motivational states are states in which instrumental goal-directed actions are performed to obtain rewards or avoid punishers, and emotional states are states that are elicited when the reward or punisher is or is not received" (Rolls, 2025)

In other words the molecular machine that triggers emotions has been constructed by our genes to pursue opportunities and to avoid threats. Bad feelings are good for us, as Randolph Nesse wrote, because they act like punishers who prevent us from suffering by threats (Nesse, 2019), and good feelings act as rewards for following opportunities. Joseph LeDoux writes that emotions are evolutionary "survival circuits" programmed to help us interact with the world. Conscious feelings (like fear, anger, or joy) occur when our higher levels detect that an unconscious survival circuit has already fired and flooded the body with chemical reactions (LeDoux, 1996)

Edmund T. Rolls writes in "Emotion Explained" (Oxford University Press, 2005) that

* Anger is triggered by the omission or termination of an expected reward
* Fear is triggered by the presentation of a threat
* Relief is triggered by the omission or termination of a threat
* Pleasure/Ecstasy is triggered by the presentation of a reward

This gives us the following matrix
```
                    PRESENTATION          OMISSION / TERMINATION
          ┌────────────────────────────┐────────────────────────────┐
  REWARD  │      PLEASURE / ECSTASY    │           ANGER            │
          └────────────────────────────┘────────────────────────────┘
  THREAT  │            FEAR            │     PLEASURE / RELIEF      │
          └────────────────────────────┘────────────────────────────┘
```
For a mere anticipation or diffuse expectation of a reward or threat we have the matrix
```
                    ANTICIPATION                  DOUBT
          ┌────────────────────────────┐────────────────────────────┐
  REWARD  │      APPETITE / DESIRE     │       FRUSTRATION          │
          └────────────────────────────┘────────────────────────────┘
  THREAT  │           ANXIETY          │         RELIEF             │
          └────────────────────────────┘────────────────────────────┘
```
Similarly we can create this matrix for elementary and moral emotions

* Pain is triggered by the violation of the physical integrity of the body
* Relief is triggered by the termination of such a violation
* Shame is triggered by the violation/non-compliance of social norms & expectations
* Pride is triggered by the restoration/compliance of social norms & expectations
  
```
                         RESTORATION                  VIOLATION     
               ┌────────────────────────────┐──────────────────────────┐
BODY INTEGRITY │      PLEASURE / RELIEF     │           PAIN           │
               └────────────────────────────┘──────────────────────────┘
SOCIAL NORMS   │          PRIDE             │       GUILT / SHAME      │
               └────────────────────────────┘──────────────────────────┘
```

Most emotions are mechanisms that are activated when something goes wrong.
Pleasure and relief is the state that all goes well. It is the subjective
experience of our own neural operating system running smoothly with zero 
friction.

As Steven Strogatz explains (Strogatz, 2014) the saddle-node bifurcation in dynamic systems is a basic mechanism for the creation and destruction of fixed points. A painful event can be described by the creation of a fixed point which represents a threat. A saddle-node model for pain with speed shading ("ghost") and the saddle's manifolds ("tipping lines") can look like this where s is the bifurcation paramter that describes the strength of the tissue damage signal:

    da/dt = −a + σ(W*a + C*s - THETA)  attention capture, self-exciting
          = −a + σ(8*a + 6*s − 7)        
    dv/dt = −v + V0 * (1 - a)          action variability, suppressed by capture
          = -v + 0.9 (1 − a)           
    σ(z) = 1 / (1 + e^(−z))            sigmoid function

In [1]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import brentq

In [2]:
W, THETA, C, V0 = 8.0, 7.0, 6.0, 0.9
C_CALM, C_PAIN, C_SADDLE, C_NEUTRAL = "#1D9E75", "#D85A30", "#7F77DD", "#8A8A85"
LIM = (0.0, 1.0)

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

def g(a, s):
    return -a + sigmoid(W * a + C * s - THETA)

def F(p, s):
    a, v = p
    return (g(a, s), -v + V0 * (1.0 - a))

def fixed_points(s, n=800):
    """Roots of g(a, s) = 0 on [0, 1] as (a*, is_stable)."""
    a = np.linspace(0.0, 1.0, n + 1)
    y = np.array([g(x, s) for x in a])
    out = []
    for i in range(n):
        if y[i] * y[i + 1] < 0:
            r = brentq(g, a[i], a[i + 1], args=(s,))
            e = 1e-6
            out.append((r, (g(r + e, s) - g(r - e, s)) / (2 * e) < 0))
    return out

def fold_points():
    out = []
    for sign in (+1, -1):
        sig = 0.5 * (1 + sign * math.sqrt(1 - 4 / W))
        z = math.log(sig / (1 - sig))
        out.append(((z - W * sig + THETA) / C, sig))
    return sorted(out)

def rk4(p, s, h):
    k1 = F(p, s)
    k2 = F((p[0] + h * k1[0] / 2, p[1] + h * k1[1] / 2), s)
    k3 = F((p[0] + h * k2[0] / 2, p[1] + h * k2[1] / 2), s)
    k4 = F((p[0] + h * k3[0], p[1] + h * k3[1]), s)
    return (p[0] + h * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0]) / 6,
            p[1] + h * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1]) / 6)

def trace(p0, s, h, n=4000, margin=0.05):
    pts = [p0]
    p = p0
    for _ in range(n):
        p = rk4(p, s, h)
        if not (LIM[0] - margin <= p[0] <= LIM[1] + margin and LIM[0] - margin <= p[1] <= LIM[1] + margin):
            break
        pts.append(p)
        if abs(F(p, s)[0]) + abs(F(p, s)[1]) < 1e-5:
            break
    return np.array(pts)

def jac(p, s, e=1e-6):
    cols = []
    for d in ((e, 0), (0, e)):
        fp = F((p[0] + d[0], p[1] + d[1]), s)
        fm = F((p[0] - d[0], p[1] - d[1]), s)
        cols.append([(fp[0] - fm[0]) / (2 * e), (fp[1] - fm[1]) / (2 * e)])
    return np.array(cols).T

def arrows(ax, pts, color, every=0.18):
    seg = np.hypot(np.diff(pts[:, 0]), np.diff(pts[:, 1]))
    cum = np.concatenate([[0], np.cumsum(seg)])
    t = every / 2
    while t < cum[-1]:
        i = min(max(np.searchsorted(cum, t), 1), len(pts) - 1)
        a, b = pts[i - 1], pts[i]
        if LIM[0] < b[0] < LIM[1] and LIM[0] < b[1] < LIM[1]:
            ax.annotate("", xy=b, xytext=a, zorder=3,
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=0,
                                        mutation_scale=8, shrinkA=0, shrinkB=0))
        t += every


In [3]:
def phase_portrait(ax, s, title):
    # 1. speed shading (dark = slow)
    gx, gy = np.meshgrid(np.linspace(*LIM, 150), np.linspace(*LIM, 150))
    sp = np.hypot(*np.vectorize(lambda x, y: F((x, y), s), otypes=[float, float])(gx, gy))
    ax.contourf(gx, gy, -np.log10(np.clip(sp, 0.01, 1.5)), levels=10, cmap="Greys", alpha=0.45, zorder=0)

    # 2. fixed points
    fps = [(a, V0 * (1 - a), stable) for a, stable in fixed_points(s)]
    fold_s, fold_a = fold_points()[0]
    merged = abs(s - fold_s) < 1e-6
    if merged:
        fps.append((fold_a, V0 * (1 - fold_a), None))
    sinks = [(a, v) for a, v, st in fps if st]

    # 3. trajectories coloured by destination
    ticks = (np.arange(10) + 0.5) / 10
    seeds = [(x, 0.0) for x in ticks] + [(x, 1.0) for x in ticks] + [(0.0, y) for y in ticks] + [(1.0, y) for y in ticks]
    seeds += [(x, y) for x in np.linspace(0.1, 0.9, 5) for y in np.linspace(0.1, 0.9, 5)]
    for s0 in seeds:
        tr = trace(s0, s, 0.05)
        end = tr[-1]
        best = min(sinks, key=lambda q: math.hypot(q[0] - end[0], q[1] - end[1]))
        col = C_PAIN if best[0] > 0.5 else C_CALM
        ax.plot(tr[:, 0], tr[:, 1], color=col, lw=0.9, alpha=0.75, zorder=2)
        arrows(ax, tr, col)

    # 4. saddle manifolds
    for a, v, st in fps:
        if st is not False:
            continue
        ev, V = np.linalg.eig(jac((a, v), s))
        vu, vs = V[:, np.argmax(ev.real)].real, V[:, np.argmin(ev.real)].real
        for sg in (-1, 1):
            p_s = (a + sg * 0.004 * vs[0], v + sg * 0.004 * vs[1])
            st_pts = trace(p_s, s, -0.05, 3000)
            ax.plot(st_pts[:, 0], st_pts[:, 1], color=C_SADDLE, lw=1.8, ls=(0, (6, 4)), zorder=4)
            p_u = (a + sg * 0.004 * vu[0], v + sg * 0.004 * vu[1])
            un_pts = trace(p_u, s, 0.05, 3000)
            ax.plot(un_pts[:, 0], un_pts[:, 1], color=C_SADDLE, lw=2.2, zorder=4)

    # 5. markers and labels
    halo = dict(edgecolor="white", linewidth=1.3, zorder=6)
    lab = dict(fontsize=9.5, fontweight="normal", zorder=8,
               bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.85))
    for a, v, st in fps:
        if st is True:
            name = "Pain" if a > 0.5 else "Calm"
            ax.scatter(a, v, s=100, color=C_PAIN if a > 0.5 else C_CALM, **halo)
            ax.text(a + (-0.04 if a > 0.5 else 0.05), v + (0.1 if a > 0.5 else 0.06), name,
                    ha="right" if a > 0.5 else "left", **lab)
        elif st is False:
            ax.scatter(a, v, s=80, color=C_SADDLE, marker="D", **halo)
            ax.text(a + 0.04, v + 0.07, "Threshold", ha="left", **lab)
        else:
            ax.scatter(a, v, s=110, facecolor=C_SADDLE, edgecolor=C_PAIN, linewidth=2.2, marker="h", zorder=6)
            ax.text(a - 0.04, v + 0.1, "Saddle-node", ha="right", **lab)

    ax.set(xlim=(-0.03, 1.03), ylim=(-0.03, 1.03), xlabel="attention capture a", ylabel="action variability v")
    ax.set_title(title, fontsize=10.5, fontweight="normal", loc="left")

def bifurcation_diagram(ax, s_now, table):
    for stab in (False, True):
        for i in range(len(table) - 1):
            (s0, r0), (s1, r1) = table[i], table[i + 1]
            for a0, st0 in r0:
                if st0 != stab:
                    continue
                cand = [(abs(a0 - a1), a1) for a1, st1 in r1 if st1 == stab]
                if cand and min(cand)[0] < 0.15:
                    a1 = min(cand)[1]
                    if stab:
                        col, lw, ls = (C_PAIN if a0 > 0.5 else C_CALM), 3.0, "-"
                    else:
                        col, lw, ls = C_SADDLE, 1.8, (0, (3, 3))
                    ax.plot([s0, s1], [a0, a1], color=col, lw=lw, ls=ls, solid_capstyle="round")
    for s_f, a_f in fold_points():
        ax.plot(s_f, a_f, "o", ms=12, mfc="none", mec="black")
        ax.annotate(f"s = {s_f:.2f}", (s_f, a_f), textcoords="offset points",
                    xytext=(10, -12) if s_f < 0.5 else (-52, 8))
    ax.axvline(s_now, color="gray", ls="--")
    for a_star, stable in fixed_points(s_now):
        ax.plot(s_now, a_star, "o", ms=9, mfc=(C_PAIN if a_star > 0.5 else C_CALM) if stable else "white",
                mec=(C_PAIN if a_star > 0.5 else C_CALM) if stable else C_SADDLE, mew=2, zorder=5)
    fold_s = fold_points()[0][0]
    if abs(s_now - fold_s) < 1e-6:
        ax.plot(s_now, fold_points()[0][1], "h", ms=11, mfc=C_SADDLE, mec=C_PAIN, mew=2, zorder=5)
    ax.set(xlim=(0, 1), ylim=(-0.03, 1.03), xlabel="damage signal s", ylabel="attention capture a*")

def make_figure(s_values, titles, path, suptitle):
    table = [(s, fixed_points(s, 800)) for s in np.linspace(0, 1, 501)]
    fig, axes = plt.subplots(len(s_values), 2, figsize=(10.5, 4.2 * len(s_values)))
    for row, s, t in zip(axes, s_values, titles):
        phase_portrait(row[0], s, t)
        bifurcation_diagram(row[1], s, table)
    handles = [
        Line2D([], [], marker="o", ls="", color=C_CALM, markersize=8, label="Calm (sink)"),
        Line2D([], [], marker="o", ls="", color=C_PAIN, markersize=8, label="Pain (sink)"),
        Line2D([], [], marker="D", ls="", color=C_SADDLE, markersize=7, label="Threshold (saddle)"),
        Line2D([], [], color=C_SADDLE, lw=1.8, ls=(0, (6, 4)), label="Tipping line (stable manifold)"),
        Line2D([], [], color=C_SADDLE, lw=2.2, label="Unstable manifold"),
        Line2D([], [], color="#777", lw=6, alpha=0.35, label="Dark shading = slow flow"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle(suptitle, fontsize=13, fontweight="normal", x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0.045, 1, 0.975))
    fig.show()
    fig.savefig(path, dpi=130)
    plt.show()      
    plt.close(fig)


In [ ]:
%matplotlib inline
make_figure(
    [0.2, 0.4, 0.6, 0.8],
    ["s = 0.2   calm only", "s = 0.4   calm and pain separated by a threshold",
     "s = 0.6   threshold close to calm", "s = 0.8   pain only"],
    "pain_saddle_node_overview.svg",
    "Pain as a saddle-node bifurcation: flow speed and tipping lines")


In [ ]:
%matplotlib inline
s_fold = fold_points()[0][0]
make_figure(
    [0.28, s_fold, 0.34, 0.40],
    ["s = 0.28   before the fold: slow ghost near a = 0.85",
     f"s = {s_fold:.3f}   critical: sink and saddle appear as one point",
     "s = 0.34   pain sink and threshold born", "s = 0.40   fully separated"],
    "pain_saddle_node_first_fold.svg",
    "Zoom on the first fold (the birth of pain)")

The saddle node divides the space in two parts: the part which contains the "calm" fix point for the "intact self", and the other part which contains the fix point for the "damaged self". The saddle is the agonizing moment of trying to inhibit or ignore the internal damage signal.

The (attention capture $a$, action variability $v$) chart is a chart of locked states. Any state that traps attention
on one object and narrows behavior will show the same geometry: two fixed points, and a saddle between them.
**Pain**, **anger**, **fear**, and **craving/desire** all fit. If we argue that qualia are the internal geometry of
homeostatic urgency, how can we distinguish between pain, anger, fear and craving? By adding more dimensions. A 2D chart is only a projection of a higher dimensional state space 
* attention capture ($a$) — [Free to Locked]
* action variability ($v$) — [Narrow to High Entropy/Exploratory]

A 3D, 4D or 5D model describes it better. A 5-dimensional state space could look like this and include goal orientation (whether the system is fleeing a boundary or hunting a target), arousal (sympathetic nervous system activation, physiological speed) and source dimension
* attention capture ($a$) — [Free to Locked]
* action variability ($v$) — [Narrow to High Entropy/Exploratory]
* goal orientation ($G$) — [Avoidance/Repulsion (-1) to Approach/Attraction (+1)]
* arousal / metabolic energy ($A$) — [Low to High] 
* domain / source dimension ($S$) — (Internal/Somatic/Tissue (0) vs. External Hazard (1) vs. Social/Reward Prediction Error (2)) which represents the categorical origin of the homeostatic violation 

The last source dimension ($S$) helps to distinguish between the four emotions which are characterized by locked states:
  
```
                       AVOID/EXPEL           APPROACH/CONSUME               DESTROY
                ┌──────────────────────┐──────────────────────────┐──────────────────────────┐
INTERNAL SOURCE │         PAIN         │            -             │            -             │
                └─────────────────────────────────────────────────┘──────────────────────────┐
BOUNDARY        │     DISGUST/NAUSEA   │      HUNGER/THIRST       │            -             │
                └─────────────────────────────────────────────────┘──────────────────────────┐
EXTERNAL SOURCE │         FEAR         │      CRAVING/DESIRE      │          ANGER           │
                └─────────────────────────────────────────────────┘──────────────────────────┘

```

Pain, anger, fear, and craving/desire are all characterized by being trapped in a locked state of high attention capture, low action variability and high arousal, but there is a difference:

* Pain: locked state plus intention to STOP INTERNAL source of discomfort
* Fear: locked state plus intention to GET AWAY from EXTERNAL source of threat
* Anger: locked state plus intention to REMOVE EXTERNAL cause of reward omission
* Desire: locked state plus intention to CONSUME EXTERNAL cause of anticipated reward

This is a useful description for the internal geometry of homeostatic survival circuits to describe qualia
because homeostasis is fundamentally about maintaining boundaries..

* Is the boundary violated? Pain
* Is the boundary threatened? Fear
* Is the boundary restored? Relief

..and exploitation of opportunitues to fill up missing resources

* Is a required resource blocked? Anger
* Is a required resource available? Desire
* Is a required resource unavailable? Frustration
* Has the required resource been obtained? Pleasure

Anticipation vs outcome:
```
                          REWARD/OPPORTUNITY              THREAT
                   ┌────────────────────────────┐───────────────────────────┐
ANTICIPATION       │     APPETITE / DESIRE      │          FEAR             │
                   └────────────────────────────┘───────────────────────────┘
PRESENCE OF CRISIS │      ANGER (if blocked)    │          PAIN             │
                   └────────────────────────────┘───────────────────────────┘
RESOLUTION FAILURE │        FRUSTRATION         │       CHRONIC PAIN        │
                   └────────────────────────────┘───────────────────────────┘
RESOLUTION SUCCESS │         PLEASURE           │         RELIEF            │
                   └────────────────────────────┘───────────────────────────┘
```

## References 

Joseph LeDoux, The emotional brain, Simon & Schuster, 1996

Randolph M. Nesse, Good Reasons for Bad Feelings: Insights from the Frontier of Evolutionary Psychiatry, Allen Lane / Penguin Books, 2019

Edmund T. Rolls, The Brain and Emotion; Oxford University Press, 1999.

Edmund T. Rolls, Emotion Explained, Oxford University Press, 2005

Edmund T. Rolls. Emotion and Decision-Making Explained; Oxford University Press, 2014.

Edmund T. Rolls, Brain Computations and Connectivity (2nd edn), Oxford University Press, 2023

Edmund T. Rolls, (2025). Emotion, Motivation, Reasoning, and How Their Brain Systems Are Related. Brain Sciences. Vol. 15. Page 507
https://www.mdpi.com/2076-3425/15/5/507

Steven Strogatz, Nonlinear Dynamics and Chaos (2nd edition), CRC Press, 2014
